# Word LLM Translation Workflow — Finalization and Output Stage

- **Workflow stage:** `finalized` (written during this notebook)
- **Input checkpoint:** fallback-completed checkpoint from the previous stage
- **Source document:** loaded from checkpoint metadata
- **Source language:** loaded from checkpoint metadata
- **Target language:** loaded from checkpoint metadata
- **Purpose of this notebook:** consolidate the completed translation state and generate final output artifacts

## Purpose of this notebook
This notebook loads the fallback-completed workflow checkpoint, verifies final translation coverage, and generates final output artifacts for review and downstream use.

## Outputs generated in this notebook
This notebook produces final artifacts in the `final_translation_files` directory, including:

- appended final Word output
- interlinear Word output
- full-schema CSV
- reduced CSV
- reduced Excel spreadsheet
- full workflow-state JSON
- reduced JSON
- Markdown parallel text
- unresolved-items report (only when unresolved items exist)

## Metadata flow
This notebook:

- loads `metadata` and `elements` from the prior checkpoint
- uses the saved workflow provenance to drive final exports
- checks for unresolved items before final export
- updates the workflow stage to `finalized`
- writes final machine-readable and human-readable output artifacts

## Notes
- Final output artifacts use the `final` field as the authoritative translated text.
- If unresolved items are present, reporting/export logic can surface them for manual review.
- This notebook does not run translation models; it packages and exports the completed workflow state.

### Setup

In [1]:
# show json files in the /checkpoints subdirectory
import importlib
import workflow_helpers
workflow_helpers = importlib.reload(workflow_helpers)

checkpoints_dir = workflow_helpers.get_checkpoints_dir()
json_files = workflow_helpers.list_json_files(checkpoints_dir)

Found JSON files:
- elements_batched_20260411_1519.json
- evaluation_completed_20260411_1604.json
- fallback_completed_20260411_1617.json
- primary_retry_completed_1_20260411_1541.json
- primary_translation_completed_20260411_1527.json


In [2]:
# Load checkpoint state (metadata + elements)
# filename contains fallback_completed

import os
from importlib import reload
import workflow_helpers
from pprint import pprint

workflow_helpers = reload(workflow_helpers)

# Define the checkpoint file
checkpoint_dir = "checkpoints"
json_file = "fallback_completed_20260411_1617.json"

# Full path
checkpoint_path = os.path.join(checkpoint_dir, json_file)

# Load normalized metadata and elements
metadata, elements = workflow_helpers.load_elements_checkpoint(checkpoint_path)

print("Loaded checkpoint:", checkpoint_path)
print("Top-level state loaded through workflow_helpers.load_elements_checkpoint")
print("\nMetadata:")
pprint(metadata)

print(f"\nLoaded checkpoint with {len(elements)} elements.")
print("\nExample entry:")
pprint(elements[0] if elements else None)

Loaded checkpoint: checkpoints\fallback_completed_20260411_1617.json
Top-level state loaded through workflow_helpers.load_elements_checkpoint

Metadata:
{'docxfilename': 'custom_word_styles_example.docx',
 'evaluation_model_name': 'gpt-5.4-mini',
 'evaluation_system_message': 'You are a translation validator for Biblical '
                              'education materials intended for a '
                              'Protestant/Evangelical audience.\n'
                              '\n'
                              'Your task is to evaluate whether each candidate '
                              'translation faithfully preserves the meaning of '
                              'the source text and preserves required Markdown '
                              '/ inline formatting.\n'
                              '\n'
                              'Be careful but not overly strict. Allow natural '
                              'translation variation. Do not fail a '
                     

In [3]:
# Reality check: final-field coverage before generating final outputs

total_elements = len(elements)
final_present = sum(bool((el.get("final") or "").strip()) for el in elements)
final_missing = total_elements - final_present

print("Total elements:", total_elements)
print("Elements with final translation:", final_present)
print("Elements missing final translation:", final_missing)

Total elements: 29
Elements with final translation: 29
Elements missing final translation: 0


In [4]:
# Set up final output directory and metadata-driven settings

from pathlib import Path
from importlib import reload
import workflow_helpers

workflow_helpers = reload(workflow_helpers)

docxfilename = metadata.get("docxfilename")
target_language = metadata.get("target_language")

missing = [
    name for name, value in {
        "docxfilename": docxfilename,
        "target_language": target_language,
    }.items()
    if not value
]

if missing:
    raise ValueError(
        f"Missing required metadata field(s) in checkpoint: {', '.join(missing)}"
    )

output_dir = Path("final_translation_files")
output_dir.mkdir(parents=True, exist_ok=True)

word_dir = Path("word_files")
template_path = word_dir / docxfilename

if not template_path.exists():
    raise FileNotFoundError(f"Template file not found: {template_path}")

print("Template DOCX:", template_path)
print("Final output directory:", output_dir)
print("Target language:", target_language)

Template DOCX: word_files\custom_word_styles_example.docx
Final output directory: final_translation_files
Target language: Traditional Chinese


In [5]:
# update stage metadata field

metadata["stage"] = workflow_helpers.WORKFLOW_STAGES["finalized"]

In [6]:
# Build finalization dataframes

df_full, df_reduced = workflow_helpers.build_finalization_dataframes(elements)

print("Full dataframe shape:", df_full.shape)
print("Reduced dataframe shape:", df_reduced.shape)

print("\nReduced dataframe preview:")
display(df_reduced.head())

Full dataframe shape: (29, 19)
Reduced dataframe shape: (29, 7)

Reduced dataframe preview:


,element_id,element_number,batch_number,text,final,final_model,is_unresolved
0,8633e724fc99,1,1,Introduction,簡介,gemini-3.1-pro-preview,False
1,2e6212d6aad5,2,1,Welcome,歡迎,gemini-3.1-pro-preview,False
2,4126aaaf767e,3,1,This course is designed for the person that wa...,本課程專為想要了解成為或作為耶穌跟隨者有何意義的人而設計。許多尋求這方面知識的人明白，他們必...,gemini-3.1-pro-preview,False
3,24ef2f4ac85e,4,2,This is a **self-directed study** to assist yo...,這是一份**自主學習**材料，旨在協助你尋找答案。它的目的是帶領你快速瀏覽精選的聖經書卷，為...,gemini-3.1-pro-preview,False
4,083f0de61472,5,3,The course consists of **two parts**: the firs...,本課程包含**兩個部分**：第一部分涵蓋基礎知識，可在兩週內完成。第二部分「深入探討」聖經，...,gemini-3.1-pro-preview,False


In [7]:
# Export appended final DOCX

appended_out_path = output_dir / workflow_helpers.build_output_path_from_base(
    docxfilename=docxfilename,
    translated_language=target_language,
    text_field="final_appended",
)

workflow_helpers.export_appended_translation_to_docx(
    elements=elements,
    template_path=template_path,
    out_path=appended_out_path,
    text_field="final",
    page_break_before_translation=True,
)

print("Saved appended final DOCX:", appended_out_path)

Saved appended final DOCX: final_translation_files\custom_word_styles_example_Traditional_Chinese_final_appended_20260411_162231.docx


In [8]:
# Export interlinear final DOCX

interlinear_out_path = output_dir / workflow_helpers.build_output_path_from_base(
    docxfilename=docxfilename,
    translated_language=target_language,
    text_field="final_interlinear",
)

workflow_helpers.export_interlinear_translation_to_docx(
    elements=elements,
    template_path=template_path,
    out_path=interlinear_out_path,
    text_field="final",
)

print("Saved interlinear final DOCX:", interlinear_out_path)

Saved interlinear final DOCX: final_translation_files\custom_word_styles_example_Traditional_Chinese_final_interlinear_20260411_162231.docx


In [9]:
# Export CSV/XLSX tables

table_paths = workflow_helpers.export_finalization_tables(
    df_full=df_full,
    df_reduced=df_reduced,
    output_dir=output_dir,
    docxfilename=docxfilename,
    target_language=target_language,
)

print("Saved tabular exports:")
print(table_paths)

Saved tabular exports:
{'full_csv': 'final_translation_files\\custom_word_styles_example_Traditional_Chinese_full_schema_20260411_162232.csv', 'reduced_csv': 'final_translation_files\\custom_word_styles_example_Traditional_Chinese_reduced_20260411_162232.csv', 'reduced_xlsx': 'final_translation_files\\custom_word_styles_example_Traditional_Chinese_reduced_20260411_162232.xlsx'}


In [11]:
# Export full final workflow-state JSON

import json
from pathlib import Path
from datetime import datetime

ts = datetime.now().strftime("%Y%m%d_%H%M%S")
base_stem = Path(docxfilename).stem

full_json_path = output_dir / f"{base_stem}_{target_language}_final_workflow_state_{ts}.json"

final_workflow_state = {
    "metadata": metadata,
    "elements": elements,
}

with open(full_json_path, "w", encoding="utf-8") as f:
    json.dump(final_workflow_state, f, ensure_ascii=False, indent=2)

print("Saved full final workflow-state JSON:", full_json_path)

Saved full final workflow-state JSON: final_translation_files\custom_word_styles_example_Traditional Chinese_final_workflow_state_20260411_162232.json


In [12]:
# Build and export reduced JSON

import json
from datetime import datetime
from pathlib import Path

ts = datetime.now().strftime("%Y%m%d_%H%M%S")
base_stem = Path(docxfilename).stem

reduced_json_path = output_dir / f"{base_stem}_{target_language}_parallel_text_reduced_{ts}.json"

reduced_json = {
    "metadata": {
        "docxfilename": metadata.get("docxfilename"),
        "source_language": metadata.get("source_language"),
        "target_language": metadata.get("target_language"),
        "stage": metadata.get("stage"),
    },
    "elements": [
        {
            "element_id": el.get("element_id"),
            "element_number": el.get("element_number"),
            "batch_number": el.get("batch_number"),
            "text": el.get("text"),
            "final": el.get("final"),
            "final_model": el.get("final_model"),
            "is_unresolved": not bool((el.get("final") or "").strip()),
        }
        for el in elements
    ],
}

with open(reduced_json_path, "w", encoding="utf-8") as f:
    json.dump(reduced_json, f, ensure_ascii=False, indent=2)

print("Saved reduced JSON:", reduced_json_path)

Saved reduced JSON: final_translation_files\custom_word_styles_example_Traditional Chinese_parallel_text_reduced_20260411_162232.json


In [13]:
# Build and export Markdown parallel text

from datetime import datetime
from pathlib import Path

ts = datetime.now().strftime("%Y%m%d_%H%M%S")
base_stem = Path(docxfilename).stem

parallel_md_path = output_dir / f"{base_stem}_{target_language}_parallel_text_{ts}.md"

lines = []
lines.append(f"# Parallel Text Export")
lines.append("")
lines.append(f"- Source file: `{docxfilename}`")
lines.append(f"- Source language: {metadata.get('source_language')}")
lines.append(f"- Target language: {metadata.get('target_language')}")
lines.append(f"- Workflow stage: {metadata.get('stage')}")
lines.append("")

for el in elements:
    element_id = el.get("element_id")
    element_number = el.get("element_number")
    batch_number = el.get("batch_number")
    source_text = el.get("text") or ""
    final_text = el.get("final") or ""
    is_unresolved = not bool(final_text.strip())

    lines.append(f"## Element {element_number}")
    lines.append("")
    lines.append(f"- element_id: `{element_id}`")
    lines.append(f"- batch_number: `{batch_number}`")
    if is_unresolved:
        lines.append(f"- status: unresolved")
    else:
        lines.append(f"- status: finalized")
    lines.append("")
    lines.append("### Source")
    lines.append("")
    lines.append(source_text)
    lines.append("")
    lines.append("### Final")
    lines.append("")
    if is_unresolved:
        lines.append("[UNRESOLVED — source text retained]")
        lines.append("")
        lines.append(source_text)
    else:
        lines.append(final_text)
    lines.append("")
    lines.append("---")
    lines.append("")

with open(parallel_md_path, "w", encoding="utf-8") as f:
    f.write("\n".join(lines))

print("Saved Markdown parallel text:", parallel_md_path)

Saved Markdown parallel text: final_translation_files\custom_word_styles_example_Traditional Chinese_parallel_text_20260411_162232.md


In [14]:
# Inspect generated final output files

generated_files = sorted(output_dir.iterdir())
print("Generated files:")
for path in generated_files:
    print(path.name)

Generated files:
custom_word_styles_example_Traditional Chinese_final_workflow_state_20260411_162232.json
custom_word_styles_example_Traditional Chinese_parallel_text_20260411_162232.md
custom_word_styles_example_Traditional Chinese_parallel_text_reduced_20260411_162232.json
custom_word_styles_example_Traditional_Chinese_final_appended_20260411_162231.docx
custom_word_styles_example_Traditional_Chinese_final_interlinear_20260411_162231.docx
custom_word_styles_example_Traditional_Chinese_full_schema_20260411_162232.csv
custom_word_styles_example_Traditional_Chinese_reduced_20260411_162232.csv
custom_word_styles_example_Traditional_Chinese_reduced_20260411_162232.xlsx


In [15]:
# Unresolved-items summary

unresolved_elements = [
    el for el in elements
    if not bool((el.get("final") or "").strip())
]

print("Unresolved element count:", len(unresolved_elements))

if unresolved_elements:
    print("\nExample unresolved item:")
    print({
        "element_id": unresolved_elements[0].get("element_id"),
        "element_number": unresolved_elements[0].get("element_number"),
        "batch_number": unresolved_elements[0].get("batch_number"),
        "text": unresolved_elements[0].get("text"),
        "primary_translation": unresolved_elements[0].get("primary_translation"),
        "fallback_translation": unresolved_elements[0].get("fallback_translation"),
        "evaluator_feedback": unresolved_elements[0].get("evaluator_feedback"),
        "fallback_error": unresolved_elements[0].get("fallback_error"),
    })
else:
    print("No unresolved elements found.")

Unresolved element count: 0
No unresolved elements found.


In [16]:
# Build unresolved-items dataframe

import pandas as pd

df_unresolved = pd.DataFrame([
    {
        "element_id": el.get("element_id"),
        "element_number": el.get("element_number"),
        "batch_number": el.get("batch_number"),
        "word_style": el.get("word_style"),
        "text": el.get("text"),
        "primary_translation": el.get("primary_translation"),
        "primary_translation_model": el.get("primary_translation_model"),
        "primary_error": el.get("primary_error"),
        "evaluator_passed": el.get("evaluator_passed"),
        "evaluator_feedback": el.get("evaluator_feedback"),
        "evaluator_error": el.get("evaluator_error"),
        "fallback_translation": el.get("fallback_translation"),
        "fallback_translation_model": el.get("fallback_translation_model"),
        "fallback_error": el.get("fallback_error"),
        "final": el.get("final"),
        "final_model": el.get("final_model"),
        "manual_translation_needed": True,
    }
    for el in unresolved_elements
])

print("Unresolved dataframe shape:", df_unresolved.shape)
display(df_unresolved.head())

Unresolved dataframe shape: (0, 0)


""


In [17]:
# Export unresolved-items CSV only if unresolved items exist

from datetime import datetime
from pathlib import Path

ts = datetime.now().strftime("%Y%m%d_%H%M%S")
base_stem = Path(docxfilename).stem

if len(df_unresolved) > 0:
    unresolved_csv_path = output_dir / f"{base_stem}_{target_language}_unresolved_items_{ts}.csv"
    df_unresolved.to_csv(unresolved_csv_path, index=False, encoding="utf-8-sig")
    print("Saved unresolved-items CSV:", unresolved_csv_path)
else:
    unresolved_csv_path = None
    print("No unresolved items to export.")

No unresolved items to export.


In [18]:
# Optional: export unresolved-items Markdown report

if unresolved_elements:
    unresolved_md_path = output_dir / f"{base_stem}_{target_language}_unresolved_items_{ts}.md"

    lines = []
    lines.append("# Unresolved Translation Items")
    lines.append("")
    lines.append(f"- Source file: `{docxfilename}`")
    lines.append(f"- Source language: {metadata.get('source_language')}")
    lines.append(f"- Target language: {metadata.get('target_language')}")
    lines.append(f"- Count: {len(unresolved_elements)}")
    lines.append("")

    for el in unresolved_elements:
        lines.append(f"## Element {el.get('element_number')}")
        lines.append("")
        lines.append(f"- element_id: `{el.get('element_id')}`")
        lines.append(f"- batch_number: `{el.get('batch_number')}`")
        lines.append(f"- word_style: `{el.get('word_style')}`")
        lines.append("")
        lines.append("### Source")
        lines.append("")
        lines.append(el.get("text") or "")
        lines.append("")
        lines.append("### Primary translation")
        lines.append("")
        lines.append(el.get("primary_translation") or "")
        lines.append("")
        lines.append("### Fallback translation")
        lines.append("")
        lines.append(el.get("fallback_translation") or "")
        lines.append("")
        lines.append("### Evaluator feedback")
        lines.append("")
        lines.append(el.get("evaluator_feedback") or "")
        lines.append("")
        lines.append("---")
        lines.append("")

    with open(unresolved_md_path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))

    print("Saved unresolved-items Markdown report:", unresolved_md_path)
else:
    print("No unresolved items to write to Markdown.")

No unresolved items to write to Markdown.


In [19]:
# Final workflow summary

final_present = sum(bool((el.get("final") or "").strip()) for el in elements)
fallback_used = sum(bool((el.get("fallback_translation") or "").strip()) for el in elements)
unresolved_count = sum(not bool((el.get("final") or "").strip()) for el in elements)

print("Finalized elements:", final_present)
print("Elements with fallback translations:", fallback_used)
print("Unresolved elements:", unresolved_count)

print("\nGenerated files:")
for path in sorted(output_dir.iterdir()):
    print("-", path.name)

Finalized elements: 29
Elements with fallback translations: 1
Unresolved elements: 0

Generated files:
- custom_word_styles_example_Traditional Chinese_final_workflow_state_20260411_162232.json
- custom_word_styles_example_Traditional Chinese_parallel_text_20260411_162232.md
- custom_word_styles_example_Traditional Chinese_parallel_text_reduced_20260411_162232.json
- custom_word_styles_example_Traditional_Chinese_final_appended_20260411_162231.docx
- custom_word_styles_example_Traditional_Chinese_final_interlinear_20260411_162231.docx
- custom_word_styles_example_Traditional_Chinese_full_schema_20260411_162232.csv
- custom_word_styles_example_Traditional_Chinese_reduced_20260411_162232.csv
- custom_word_styles_example_Traditional_Chinese_reduced_20260411_162232.xlsx


## Notebook checkpoint and output summary

This notebook completed the finalization stage of the Word-to-LLM workflow by performing the following steps:

- loaded the fallback-completed checkpoint containing metadata and fully processed element state
- verified final translation coverage across all elements
- confirmed whether any unresolved items remained before export
- generated final Word output by appending the finalized translation to the source document
- generated an interlinear Word output alternating source and finalized translation by element
- built full and reduced tabular views of the workflow state
- exported full and reduced tabular outputs to CSV, and reduced output to Excel
- exported a full final workflow-state JSON artifact
- exported a reduced JSON artifact for lighter machine-readable use
- exported a Markdown parallel-text artifact for lightweight review
- generated unresolved-item reporting logic for future runs with residual failures
- updated workflow metadata to reflect the `finalized` stage
- wrote all final artifacts to the `final_translation_files` directory

### Output of this notebook
The main outputs of this notebook are final review and delivery artifacts derived from the completed workflow state.

These may include:

- appended final `.docx`
- interlinear `.docx`
- full-schema `.csv`
- reduced `.csv`
- reduced `.xlsx`
- full workflow-state `.json`
- reduced `.json`
- parallel-text `.md`
- unresolved-items report files when applicable

### Metadata saved at this stage
The final workflow-state metadata records the full provenance accumulated across the pipeline, including:

- `stage = finalized`
- `docxfilename`
- `source_language`
- `target_language`
- `primary_model_name`
- `evaluation_model_name`
- `fallback_model_name`
- `primary_system_message`
- `evaluation_system_message`
- `fallback_system_message`

### Scope of this notebook
This notebook focuses on final packaging and export only.

At this stage:

- no new translation is generated
- no new evaluation is performed
- the `final` field is treated as the authoritative translation output
- unresolved-item handling remains available for larger or more difficult documents
- final artifacts are generated in multiple formats for human review, spreadsheet use, and machine-readable provenance

### End state
This notebook closes out the current Word translation workflow run. The exported artifacts in `final_translation_files` represent the finalized outputs for this document.